<a href="https://colab.research.google.com/github/VanAgostine/BTE320/blob/main/MatteoAgostine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
from PyQt6.QtWidgets import (QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
                            QPushButton, QLabel, QLineEdit,
                            QMessageBox, QTabWidget, QDialog, QDialogButtonBox)
from PyQt6.QtGui import QFont
from PyQt6.QtCore import Qt

from . import styles


class MainWindow(QMainWindow):

    def __init__(self):
        super().__init__()
        self.chart_tabs = {}
        self.dark_theme = True
        self.first_load = True

        self.setup_ui()
        self.apply_theme()

    def setup_ui(self):
        self.setWindowTitle("Stock Chart Viewer")
        self.setGeometry(100, 100, 1400, 800)

        central_widget = QWidget()
        self.setCentralWidget(central_widget)

        self.main_layout = QVBoxLayout(central_widget)
        self.main_layout.setSpacing(8)
        self.main_layout.setContentsMargins(8, 8, 8, 8)

        self.center_widget = QWidget()
        center_layout = QVBoxLayout(self.center_widget)
        center_layout.setAlignment(Qt.AlignmentFlag.AlignCenter)

        input_container = QWidget()
        input_container.setMaximumWidth(650)
        input_layout = QHBoxLayout(input_container)
        input_layout.setSpacing(8)
        input_layout.setContentsMargins(0, 0, 0, 0)

        ticker_label = QLabel("Enter Ticker:")
        ticker_font = QFont("Courier New", 14)
        ticker_font.setBold(True)
        ticker_font.setLetterSpacing(QFont.SpacingType.AbsoluteSpacing, 2)
        ticker_label.setFont(ticker_font)
        input_layout.addWidget(ticker_label)

        self.ticker_input = QLineEdit()
        self.ticker_input.setFixedWidth(120)
        self.ticker_input.setMinimumHeight(40)
        self.ticker_input.returnPressed.connect(self.load_chart)
        input_layout.addWidget(self.ticker_input)

        self.load_button = QPushButton("Load Chart")
        self.load_button.clicked.connect(self.load_chart)
        self.load_button.setMinimumHeight(40)
        self.load_button.setMinimumWidth(120)
        self.load_button.setToolTip("Load chart for ticker (Enter)")
        input_layout.addWidget(self.load_button)

        self.csv_button = QPushButton("Load CSV Data")
        self.csv_button.clicked.connect(self.load_csv_data)
        self.csv_button.setMinimumHeight(40)
        self.csv_button.setMinimumWidth(140)
        self.csv_button.setToolTip("Load CSV data for MES or MNQ")
        input_layout.addWidget(self.csv_button)

        center_layout.addWidget(input_container, 0, Qt.AlignmentFlag.AlignCenter)
        self.main_layout.addWidget(self.center_widget)

        self.toolbar_widget = QWidget()
        self.toolbar_layout = self.create_toolbar()
        self.toolbar_widget.setLayout(self.toolbar_layout)
        self.toolbar_widget.hide()

        self.tab_widget = QTabWidget()
        self.tab_widget.setTabsClosable(True)
        self.tab_widget.tabCloseRequested.connect(self.close_tab)
        self.tab_widget.currentChanged.connect(self.on_tab_switched)
        self.tab_widget.currentChanged.connect(self.update_close_button_colors)
        self.tab_widget.hide()

    def create_toolbar(self):
        toolbar_layout = QHBoxLayout()

        ticker_label = QLabel("Ticker:")
        ticker_label.setFont(QFont("Arial", 10, QFont.Weight.Bold))
        toolbar_layout.addWidget(ticker_label)

        self.toolbar_ticker_input = QLineEdit()
        self.toolbar_ticker_input.setFixedWidth(120)
        self.toolbar_ticker_input.setMinimumHeight(35)
        self.toolbar_ticker_input.returnPressed.connect(self.load_chart_from_toolbar)
        toolbar_layout.addWidget(self.toolbar_ticker_input)

        self.toolbar_load_button = QPushButton("Load Chart")
        self.toolbar_load_button.clicked.connect(self.load_chart_from_toolbar)
        self.toolbar_load_button.setMinimumHeight(35)
        self.toolbar_load_button.setMinimumWidth(100)
        self.toolbar_load_button.setToolTip("Load chart for ticker (Enter)")
        toolbar_layout.addWidget(self.toolbar_load_button)

        self.toolbar_csv_button = QPushButton("Load CSV")
        self.toolbar_csv_button.clicked.connect(self.load_csv_data)
        self.toolbar_csv_button.setMinimumHeight(35)
        self.toolbar_csv_button.setMinimumWidth(100)
        self.toolbar_csv_button.setToolTip("Load CSV data for MES or MNQ")
        toolbar_layout.addWidget(self.toolbar_csv_button)

        toolbar_layout.addStretch()

        return toolbar_layout



    def apply_theme(self):
        if self.dark_theme:
            self.setStyleSheet(styles.DARK_THEME)
        else:
            self.setStyleSheet(styles.LIGHT_THEME)

    def load_chart(self):
        ticker = self.ticker_input.text().strip().upper()

        if not ticker:
            QMessageBox.warning(self, "No Ticker", "Please enter a ticker symbol.")
            return

        if self.first_load:
            self.center_widget.hide()
            self.main_layout.insertWidget(0, self.toolbar_widget)
            self.toolbar_widget.show()
            self.main_layout.addWidget(self.tab_widget)
            self.tab_widget.show()
            self.first_load = False

        self.open_chart_tab(ticker)
        self.ticker_input.clear()
        self.ticker_input.setFocus()

    def load_chart_from_toolbar(self):
        ticker = self.toolbar_ticker_input.text().strip().upper()

        if not ticker:
            QMessageBox.warning(self, "No Ticker", "Please enter a ticker symbol.")
            return

        self.open_chart_tab(ticker)
        self.toolbar_ticker_input.clear()
        self.toolbar_ticker_input.setFocus()

    def load_csv_data(self):
        dialog = QDialog(self)
        dialog.setWindowTitle("Select Ticker")
        dialog.setFixedSize(250, 120)

        layout = QVBoxLayout(dialog)

        label = QLabel("Select ticker for CSV data:")
        layout.addWidget(label)

        button_layout = QHBoxLayout()
        mes_button = QPushButton("MES")
        mnq_button = QPushButton("MNQ")
        mes_button.setMinimumHeight(35)
        mnq_button.setMinimumHeight(35)
        button_layout.addWidget(mes_button)
        button_layout.addWidget(mnq_button)
        layout.addLayout(button_layout)

        selected_ticker = [None]

        def select_mes():
            selected_ticker[0] = "MES"
            dialog.accept()

        def select_mnq():
            selected_ticker[0] = "MNQ"
            dialog.accept()

        mes_button.clicked.connect(select_mes)
        mnq_button.clicked.connect(select_mnq)

        if dialog.exec() != QDialog.DialogCode.Accepted or selected_ticker[0] is None:
            return

        ticker = selected_ticker[0]
        base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        csv_path = os.path.join(base_dir, "data", f"{ticker}_v0_ohlcv_1m.csv")

        if not os.path.exists(csv_path):
            QMessageBox.warning(self, "File Not Found", f"CSV file not found:\n{csv_path}")
            return

        try:
            csv_data = pd.read_csv(csv_path, skipinitialspace=True)
            csv_data.columns = csv_data.columns.str.strip()
            csv_data['datetime'] = pd.to_datetime(csv_data['date'] + ' ' + csv_data['time'])
            csv_data.set_index('datetime', inplace=True)
            csv_data.rename(columns={
                'open': 'Open',
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'volume': 'Volume'
            }, inplace=True)
            csv_data = csv_data[['Open', 'High', 'Low', 'Close', 'Volume']]
        except Exception as e:
            QMessageBox.warning(self, "CSV Error", f"Failed to load CSV:\n{str(e)}")
            return

        if self.first_load:
            self.center_widget.hide()
            self.main_layout.insertWidget(0, self.toolbar_widget)
            self.toolbar_widget.show()
            self.main_layout.addWidget(self.tab_widget)
            self.tab_widget.show()
            self.first_load = False

        self.open_csv_chart_tab(ticker, csv_data)

    def open_csv_chart_tab(self, ticker, csv_data):
        tab_key = f"{ticker} - CSV"

        if tab_key in self.chart_tabs:
            chart_widget = self.chart_tabs[tab_key]
            idx = self.tab_widget.indexOf(chart_widget)
            if idx != -1:
                self.tab_widget.setCurrentIndex(idx)
                return
            else:
                del self.chart_tabs[tab_key]

        try:
            from .chart_widget import ChartWidget
        except ImportError as e:
            QMessageBox.warning(self, "Cannot open chart", f"Missing chart dependencies:\n{e}")
            return

        chart_widget = ChartWidget(ticker, csv_data=csv_data)
        chart_widget.ticker_changed.connect(self.on_ticker_changed)
        tab_index = self.tab_widget.addTab(chart_widget, tab_key)
        self.tab_widget.setCurrentIndex(tab_index)
        self.chart_tabs[tab_key] = chart_widget
        self.update_close_button_colors()

    def open_chart_tab(self, ticker: str):
        ticker = ticker.upper()

        if ticker in self.chart_tabs:
            chart_widget = self.chart_tabs[ticker]
            idx = self.tab_widget.indexOf(chart_widget)
            if idx != -1:
                self.tab_widget.setCurrentIndex(idx)
                return
            else:
                del self.chart_tabs[ticker]

        try:
            from .chart_widget import ChartWidget
        except ImportError as e:
            QMessageBox.warning(self, "Cannot open chart", f"Missing chart dependencies:\n{e}")
            return

        chart_widget = ChartWidget(ticker)
        chart_widget.ticker_changed.connect(self.on_ticker_changed)
        tab_label = f"{ticker}"
        tab_index = self.tab_widget.addTab(chart_widget, tab_label)
        self.tab_widget.setCurrentIndex(tab_index)
        self.chart_tabs[ticker] = chart_widget
        self.update_close_button_colors()

    def on_ticker_changed(self, old_ticker, new_ticker):
        if old_ticker in self.chart_tabs:
            chart_widget = self.chart_tabs[old_ticker]
            del self.chart_tabs[old_ticker]
            self.chart_tabs[new_ticker] = chart_widget

            idx = self.tab_widget.indexOf(chart_widget)
            if idx != -1:
                self.tab_widget.setTabText(idx, new_ticker)

    def close_tab(self, index):
        widget = self.tab_widget.widget(index)

        for ticker, chart_widget in list(self.chart_tabs.items()):
            if chart_widget is widget:
                del self.chart_tabs[ticker]
                break

        self.tab_widget.removeTab(index)

    def on_tab_switched(self, index):
        pass

    def update_close_button_colors(self):
        tab_bar = self.tab_widget.tabBar()
        current_index = self.tab_widget.currentIndex()

        for i in range(self.tab_widget.count()):
            close_button = tab_bar.tabButton(i, tab_bar.ButtonPosition.RightSide)
            if close_button:
                if i == current_index:
                    close_button.setStyleSheet("""
                        QAbstractButton {
                            background-color: #000000;
                            border: none;
                            border-radius: 2px;
                        }
                        QAbstractButton:hover {
                            background-color: #333333;
                        }
                    """)
                else:
                    close_button.setStyleSheet("""
                        QAbstractButton {
                            background-color: #ff8c00;
                            border: none;
                            border-radius: 2px;
                        }
                        QAbstractButton:hover {
                            background-color: #ff6600;
                        }
                    """)

    def closeEvent(self, event):
        event.accept()


ModuleNotFoundError: No module named 'PyQt6'